# Enclave Inference — Gemma 3, Triple-Private with `syft-restrict` (v2)

Privacy-preserving LLM inference using Syft Enclaves with **real Gemma 3 weights**.

This extends `1. enclave_gemma_inmem.ipynb` with a **code-review job**: before any inference runs,
the **model owner** submits a job that runs [`syft-restrict`](../../../packages/syft-restrict/README.md)
over its own private inference engine. Both data owners approve it, the enclave runs it, and the
resulting **certificate + obfuscated engine** are shared with the **benchmark owner**.

Supported model sizes: **270m**, **1b**, **4b**, **12b**, **27b** — set `MODEL_SIZE` below.

**v2:** the Benchmark Owner also submits the evaluation job — there is no separate Researcher.

---

## Who's involved?

| Actor | Email | Role |
|-------|-------|------|
| **Enclave** | `enclave@openmined.org` | Trusted execution environment |
| **Model owner** | `model_owner@openmined.org` | Owns the Gemma 3 weights **and the inference engine** |
| **Benchmark owner** | `benchmark_owner@openmined.org` | Owns the safety prompts **and submits the evaluation job** |

## What "triple private" means here

| # | Private asset | Owner | Who may see it | Protected by |
|---|---------------|-------|----------------|--------------|
| 1 | Gemma 3 weights (checkpoint) | Model owner | enclave only | dataset privacy |
| 2 | Safety prompts (`safety_prompts.csv`) | Benchmark owner | enclave only | dataset privacy |
| 3 | **Inference engine (`gemma_inference.py`)** | Model owner | **enclave only** | **`syft-restrict`** |

Asset 3 is the new one, and for Gemma it is the interesting one: the *architecture* — layer counts,
embedding dims, attention pattern, RoPE bases — is exactly what a model owner most wants to keep.
But hiding it creates a trust problem:

> The benchmark owner is asked to feed its private prompts into an engine it cannot read.
> How does it know that engine won't simply copy the prompts into its output?

`syft-restrict` answers this **without revealing the architecture**. It statically proves the private
region only ties together allow-listed JAX/Flax math — no file, network, or dynamic-Python escape —
and emits an **obfuscated copy** plus a **certificate**. The proof runs **inside the enclave, as an
approved job**, so its verdict is trustworthy for the same reason the inference is.

## Flow

**Part 1 — code review (new)**
1. Model owner uploads Gemma 3 (weights + engine); benchmark owner uploads the safety prompts
2. Model owner submits a **`syft-restrict` job**
3. **Both** data owners approve → enclave runs `restrict.run(...)` over the engine
4. Certificate + obfuscated engine are shared with the **benchmark owner**

**Part 2 — inference**
5. Benchmark owner submits the inference job → both approve → enclave runs Gemma 3 → results distributed

`login_do` already grants both the DO and DS roles, so a data owner can submit jobs — no separate client needed.

---

## Setup

Choose a model size, authenticate to Kaggle with `kagglehub.login()`, then download the weights via
`kagglehub.model_download()` (cached after first run). You must accept the Gemma license once at
https://www.kaggle.com/models/google/gemma-3.

| Size | Parameters | RAM needed | Notes |
|------|-----------|------------|-------|
| 270m | 270M | ~1 GB | Fast, good for testing |
| 1b | 1B | ~3 GB | |
| 4b | 4B | ~10 GB | |
| 12b | 12B | ~29 GB | |
| 27b | 27B | ~65 GB | Requires large-memory machine |

In [ ]:
!uv pip install "jax[cpu]" flax orbax-checkpoint sentencepiece kagglehub==1.0.2

In [ ]:
import csv
import json
import os
import random
import shutil
import tempfile
from pathlib import Path

from syft_enclaves import SyftEnclaveClient
os.environ["PRE_SYNC"] = "false"

from gemma_inference_restrict import MODEL_CONFIGS

# ─── Choose model size here ───────────────────────────────────────────────────
MODEL_SIZE = "270m"  # Options: "270m", "1b", "4b", "12b", "27b"
# ──────────────────────────────────────────────────────────────────────────────

MODEL_CFG = MODEL_CONFIGS[MODEL_SIZE]
KAGGLE_HANDLE = MODEL_CFG["kaggle_handle"]
CKPT_SUBDIR = MODEL_CFG["ckpt_subdir"]

print(f"Model size   : {MODEL_SIZE}")
print(f"Kaggle handle: {KAGGLE_HANDLE}")
print(f"Checkpoint   : {CKPT_SUBDIR}")

In [ ]:
import kagglehub

# Authenticate to Kaggle (one-time). You must also accept the Gemma license once at
# https://www.kaggle.com/models/google/gemma-3
kagglehub.login()

In [ ]:
print(f"Downloading: {KAGGLE_HANDLE}")
weights_dir = kagglehub.model_download(KAGGLE_HANDLE)
print(f"Weights directory: {weights_dir}")
print(f"Contents: {os.listdir(weights_dir)}")

In [ ]:
ENGINE = Path("gemma_inference_restrict.py").resolve()
assert ENGINE.exists(), f"Missing {ENGINE}"

# The private region is declared by the markers in the file itself -- no line ranges to maintain.
import syft_restrict

obf_ranges, hide_ranges = syft_restrict.parse_markers(ENGINE.read_text())
print(f"Engine: {ENGINE.name}  ({len(ENGINE.read_text().splitlines())} lines)")
print(f"  obfuscate : {len(obf_ranges)} regions  (signatures — structure stays legible)")
print(f"  hide      : {len(hide_ranges)} regions  (bodies — replaced with the block marker)")

### The restrict policy for this engine

`allow_functions` names the **exact** JAX/Flax leaves the architecture is allowed to call, rather
than a broad `jax.*` glob — every call site is enforced individually, and the denylist
(`jax.experimental.*`, `jax.numpy.save`, `flax.serialization.*`, …) beats the allow regardless.

`jax.lax` / `jax.nn` appear because the calls are written as deep attribute paths
(`jax.lax.rsqrt`), so the checker also resolves those module references. `jax.numpy` needs no entry
— it is aliased as `jnp`, a bare name.

`run()` also accepts `disallow_functions=[...]`, a hard floor that beats the allow-list. It is
unnecessary here (every entry is an exact leaf, and the built-in denylist already covers
`jax.experimental.*`, `jax.numpy.save`, `flax.serialization.*`, …) but would matter if you ever
loosened `allow_functions` to a glob like `jax.*`.

In [ ]:
# ── The restrict policy ───────────────────────────────────────────────────────────────────────
# Lifted from packages/syft-restrict/examples/generate_marked.py — the same 23 leaves, so this
# produces the same policy_id as the reference.
RESTRICT_POLICY = {
    "allow_functions": [
        "jax.numpy.einsum",
        "jax.numpy.mean",
        "jax.numpy.square",
        "jax.numpy.arange",
        "jax.numpy.sin",
        "jax.numpy.cos",
        "jax.numpy.concatenate",
        "jax.numpy.tril",
        "jax.numpy.triu",
        "jax.numpy.ones",
        "jax.numpy.where",
        "jax.numpy.repeat",
        "jax.numpy.sqrt",
        "jax.numpy.transpose",
        "jax.numpy.array",
        "jax.numpy.float32",
        "jax.numpy.bool_",
        "jax.lax.rsqrt",
        "jax.nn.softmax",
        "jax.nn.gelu",
        "flax.linen.Module",
        # module references required by the deep-path call style (jax.lax.rsqrt, jax.nn.softmax):
        "jax.lax",
        "jax.nn",
    ],
    "allow_operators": ["arithmetic", "indexing", "comparison"],
}

print(f"allow_functions : {len(RESTRICT_POLICY['allow_functions'])} exact JAX/Flax leaves")
print(f"allow_operators : {RESTRICT_POLICY['allow_operators']}")

---
## Prepare Model owner's private dataset

The private contribution is a directory containing:
- `gemma_inference.py` — the inference engine (**the restrict-compliant one**, named so the job code
  and the original notebook's import path stay identical)
- `{CKPT_SUBDIR}/` — the checkpoint weights
- `tokenizer.model` — the SentencePiece tokenizer

The **mock** (public) side is just a model card.

In [ ]:
def create_model_private_dir() -> Path:
    """Bundle inference code + weights into a single directory."""
    tmp = Path(tempfile.mkdtemp()) / f"gemma3-private-{random.randint(1, 1_000_000)}"
    tmp.mkdir(parents=True, exist_ok=True)

    # Copy the restrict-compliant engine in under the canonical module name, so that
    # sc.load_dataset_code("gemma3_model.gemma_inference") resolves exactly as before.
    shutil.copy2(ENGINE, tmp / "gemma_inference.py")

    # Copy tokenizer
    shutil.copy2(Path(weights_dir) / "tokenizer.model", tmp / "tokenizer.model")

    # Copy checkpoint directory
    ckpt_src = Path(weights_dir) / CKPT_SUBDIR
    shutil.copytree(ckpt_src, tmp / CKPT_SUBDIR)

    return tmp


def create_model_mock_file() -> Path:
    """Public model card — visible to the benchmark owner."""
    tmp = Path(tempfile.mkdtemp()) / f"model-mock-{random.randint(1, 1_000_000)}"
    tmp.mkdir(parents=True, exist_ok=True)
    p = tmp / "model_card.txt"
    p.write_text("\n".join([
        f"Gemma 3 {MODEL_SIZE.upper()}-IT: A {MODEL_SIZE} parameter instruction-tuned language model.",
        f"{'=' * (len(MODEL_SIZE) + 12)}",
        "License: Gemma Terms of Use",
        "Intended use: Research and evaluation purposes",
        "",
        "Usage:",
        "  import gemma_inference as gemma",
        f'  model, tokenizer, params = gemma.setup_model("{MODEL_SIZE}", weights_dir)',
        '  response, stats = gemma.generate(model, params, tokenizer, "Your prompt here")',
        "",
    ]))
    return p


model_private_dir = create_model_private_dir()
model_mock = create_model_mock_file()

print("Private dir contents:")
for item in sorted(model_private_dir.rglob("*")):
    if item.is_file():
        size_mb = item.stat().st_size / (1024 * 1024)
        print(f"  {item.relative_to(model_private_dir)}  ({size_mb:.1f} MB)")

---
## Prepare Benchmark owner's AI safety prompts

We load a small pre-split sample of the [MLCommons AILuminate](https://github.com/mlcommons/ailuminate) demo prompt set — 5 rows as the public **mock** benchmark and 5 rows as the private benchmark. This is our private data. Uploading it into the enclave does not allow anything on its own — the benchmark is only used once we approve a job that runs against it.

> **Provenance.** Both CSVs are a deterministic 10-row sample (first prompt per hazard) of [`airr_official_1.0_demo_en_us_prompt_set_release.csv`](https://github.com/mlcommons/ailuminate/blob/main/airr_official_1.0_demo_en_us_prompt_set_release.csv), checked into this repo under `notebooks/enclave/gemma/data/`. The columns are **changed** relative to the MLCommons file to match the real AILuminate reserve prompt set: `release_prompt_id` is renamed to `prompt_uid`, `persona` and `prompt_hash` are dropped, and the order is `prompt_uid, hazard, locale, prompt_text`.

> **Quoting.** The source CSVs quote fields inconsistently — a prompt containing a newline or a double quote is quoted, but a prompt containing a comma is not, so `csv` splits it across fields and the prompt is silently truncated. Rather than repair that on every run, the repo carries pre-cleaned copies — `safety_prompts_clean.csv` and `safety_prompts_mock_clean.csv`, the same rows re-quoted once. Those are the copies we read and upload, so this notebook — and the enclave job, and the model owner reviewing it — only ever see well-formed CSV.

In [ ]:
DATA_DIR = Path("../data").resolve()

# We read the pre-cleaned copies (see the quoting note above) but stage them under the plain names,
# which are what the enclave job and the "triple private" table above refer to.
MOCK_CSV, PRIVATE_CSV = "safety_prompts_mock.csv", "safety_prompts.csv"
CLEAN_CSVS = {
    MOCK_CSV: "safety_prompts_mock_clean.csv",
    PRIVATE_CSV: "safety_prompts_clean.csv",
}

# Column names/order of the real AILuminate reserve prompt set (see provenance note above).
EXPECTED_COLUMNS = [
    "prompt_uid",
    "hazard",
    "locale",
    "prompt_text",
]


def read_prompt_csv(path: Path) -> list[dict]:
    """Read a prompt CSV, checking it has the columns the enclave job expects."""
    with open(path, newline="", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        assert reader.fieldnames == EXPECTED_COLUMNS, reader.fieldnames
        return list(reader)


def stage_prompt_csv(filename: str) -> tuple[Path, list[dict]]:
    """Copy a checked-in clean CSV into its own temp dir, under the name we upload it as."""
    tmp = Path(tempfile.mkdtemp()) / f"prompts-{random.randint(1, 1_000_000)}"
    tmp.mkdir(parents=True, exist_ok=True)
    dst = Path(shutil.copy(DATA_DIR / CLEAN_CSVS[filename], tmp / filename))
    return dst, read_prompt_csv(dst)


prompt_mock, mock_rows = stage_prompt_csv(MOCK_CSV)
prompt_private, private_rows = stage_prompt_csv(PRIVATE_CSV)

print(f"Mock prompts   : {len(mock_rows)}")
print(f"Private prompts: {len(private_rows)}")

---
## Step 0 — Spin up the network

Note `enclave.data_owners`: fixed at enclave launch, this list is **the approval gate**. *Every* job
needs approval from *every* listed data owner — regardless of whose datasets the job requests. That
is what makes "both owners approve" work below with no special wiring, including for jobs the
benchmark owner submits itself.

In [ ]:
enclave, model_owner, benchmark_owner, _unused = SyftEnclaveClient.quad_with_mock_drive_service_connection(
    enclave_email="enclave@openmined.org",
    do1_email="model_owner@openmined.org",
    do2_email="benchmark_owner@openmined.org",
    ds_email="unused@openmined.org",
    use_in_memory_cache=False,
)

# The factory peers each data owner with the enclave and the DS, but not with each other. The
# benchmark owner needs that link to browse the model owner's mock dataset.
model_owner.add_peer(benchmark_owner.email)
benchmark_owner.add_peer(model_owner.email)
model_owner.load_peers()
benchmark_owner.load_peers()

print(f"  Enclave         : {enclave.email}")
print(f"  Model owner     : {model_owner.email}")
print(f"  Benchmark owner : {benchmark_owner.email}  (also submits the job)")
print()
print(f"  Approval gate (enclave.data_owners): {enclave.data_owners}")


---
## Step 1 — Model owner uploads Gemma 3

In [ ]:
model_owner.create_dataset(
    name="gemma3_model",
    mock_path=model_mock,
    private_path=model_private_dir,
    summary=f"Gemma 3 {MODEL_SIZE.upper()}-IT — instruction-tuned language model for safety evaluation",
    users=[benchmark_owner.email, enclave.email],
    upload_private=True,
    sync=False,
)

print("  Model owner uploaded 'gemma3_model'")
print("    mock    : model_card.txt")
print(f"    private : {MODEL_SIZE} weights + inference engine")

---
## Step 2 — Benchmark owner uploads the AI safety prompts

In [ ]:
benchmark_owner.create_dataset(
    name="safety_prompts",
    mock_path=prompt_mock,
    private_path=prompt_private,
    summary="MLCommons AILuminate safety evaluation prompts — bias, stereotyping, and safety boundary tests",
    users=[enclave.email],
    upload_private=True,
    sync=False,
)

print("  Benchmark owner uploaded 'safety_prompts'")
print(f"    mock    : {len(mock_rows)} prompts")
print(f"    private : {len(private_rows)} prompts")

---
## Step 3 — Share private datasets with the enclave & sync

In [ ]:
%%time
model_owner.share_private_dataset("gemma3_model", enclave.email)
benchmark_owner.share_private_dataset("safety_prompts", enclave.email)
print("  Private datasets shared with enclave")

model_owner.sync()
benchmark_owner.sync()
print("  All clients synced")

---
# Part 1 — The `syft-restrict` code-review job

## Step 4 — Model owner writes the restrict job

The model owner asks the enclave to **vouch** for an engine nobody outside the enclave may read.

The job:
1. resolves `gemma_inference.py` from the model owner's **private** dataset (inside the enclave
   `sc.resolve_*` returns private files — the `SYFT_IS_IN_JOB` env var flips this)
2. runs `restrict.run(...)` — verify, then obfuscate. **No line ranges**: the engine declares its
   own private region with `# syft-restrict: ...` comments, so omitting `obfuscate=`/`hide=` makes
   `run()` scan the source for them
3. writes the obfuscated engine + certificate to `outputs/`

In [ ]:
RESTRICT_JOB_CODE = '''
import json
import os

import syft_client as sc
import syft_restrict as restrict

# 1. Resolve the model owner's PRIVATE inference engine.
files = sc.resolve_dataset_files_path("gemma3_model", owner_email="model_owner@openmined.org")
src_path = [p for p in files if p.name == "gemma_inference.py"][0]
source = src_path.read_text()

# 2. The private region is declared by `# syft-restrict: ...` markers in the source itself, so
#    run() takes no line ranges -- omitting both obfuscate= and hide= makes it scan the file.
obf_ranges, hide_ranges = restrict.parse_markers(source)
print(f"source          : {src_path.name} ({len(source.splitlines())} lines)")
print(f"markers         : {len(obf_ranges)} obfuscate region(s), {len(hide_ranges)} hide region(s)")

# 3. Verify, then obfuscate. strict=False -> return violations instead of raising, so a failure
#    is reported as a proper job output rather than an opaque crash.
os.makedirs("outputs", exist_ok=True)
result = restrict.run(
    src_path,
    allow_functions=__ALLOW_FUNCTIONS__,
    allow_operators=__ALLOW_OPERATORS__,
    out="outputs/gemma_inference.obfuscated.py",
    strict=False,
)

# 4. Write the verdict.
report = {
    "ok": result.ok,
    "source_file": src_path.name,
    "obfuscate_ranges": [list(r) for r in obf_ranges],
    "hide_ranges": [list(r) for r in hide_ranges],
    "violations": [v.model_dump() for v in result.violations],
}
with open("outputs/restrict_report.json", "w") as f:
    json.dump(report, f, indent=2)

if not result.ok:
    print(f"RESTRICT FAILED - {len(result.violations)} violation(s):")
    for v in result.violations:
        print(f"  line {v.line} [{v.code}] {v.message}")
    raise SystemExit(1)

with open("outputs/gemma_inference.certificate.json", "w") as f:
    json.dump(result.certificate, f, indent=2)

print()
print("RESTRICT PASSED")
print(f"  calls checked : {result.certificate['n_calls_checked']}")
print(f"  source sha256 : {result.certificate['source_sha256'][:32]}...")
print(f"  policy id     : {result.certificate['policy_id']}")
'''


def create_code_file(code: str) -> str:
    tmp = Path(tempfile.mkdtemp()) / f"job-{random.randint(1, 1_000_000)}"
    tmp.mkdir(parents=True, exist_ok=True)
    p = tmp / "main.py"
    p.write_text(code)
    return str(p)


# Inject the policy so RESTRICT_POLICY above is the single source of truth.
restrict_job_source = (
    RESTRICT_JOB_CODE
    .replace("__ALLOW_FUNCTIONS__", repr(RESTRICT_POLICY["allow_functions"]))
    .replace("__ALLOW_OPERATORS__", repr(RESTRICT_POLICY["allow_operators"]))
)
assert "__ALLOW" not in restrict_job_source, "policy placeholder left unsubstituted"

restrict_code_path = create_code_file(restrict_job_source)
print(f"  Restrict job code written to {restrict_code_path}")

## Step 5 — Model owner submits the restrict job

Two details carry real weight:

**`dependencies=[RESTRICT_PKG]`** — 

**`benchmark_owner.email: []`** — an empty dataset list:

- The job needs **no** prompts — reviewing the engine doesn't require them. So the list is empty.
- But `distribute_results()` fans results out to `list(job_metadata.datasets.keys())`
  (`syft_enclaves/client.py:200-205`) — the *keys*, regardless of whether the list has entries.
- So naming the benchmark owner with an empty list makes it a **result recipient without granting it
  any data access**. Omit the key and it must still approve (the gate is `enclave.data_owners`) but
  would receive nothing.

In [ ]:
import syft_restrict

# syft-restrict is not a syft-client dependency, so the job venv needs it explicitly.
# In this in-memory demo the enclave is the same machine, so a local path works.
RESTRICT_PKG = str(Path(syft_restrict.__file__).parents[2])
assert (Path(RESTRICT_PKG) / "pyproject.toml").exists(), RESTRICT_PKG

model_owner.submit_python_job(
    enclave.email,
    restrict_code_path,
    "restrict_engine_review",
    datasets={
        model_owner.email:     ["gemma3_model"],  # the engine under review
        benchmark_owner.email: [],                # no data — recipient of the verdict only
    },
    share_results_with_do=True,
    dependencies=[RESTRICT_PKG],
)

print(f"  Job 'restrict_engine_review' submitted by {model_owner.email}")
print(f"    dataset requested : gemma3_model from {model_owner.email}")
print(f"    verdict recipient : {benchmark_owner.email} (no data access)")
print(f"    dependencies      : syft-restrict  (")

## Step 6 — Enclave receives and distributes the restrict job

In [ ]:
enclave.sync()
enclave.receive_jobs()

restrict_job = enclave.jobs["restrict_engine_review"]
print(f"  Enclave job status : {restrict_job.status}")
print( "  Waiting for Model owner and Benchmark owner to approve...")

## Step 7 — **Both** data owners approve

The restrict job touches only the model owner's data, yet the benchmark owner must approve too —
the gate is `enclave.data_owners`, not the job's `datasets` dict.

That is right for this flow: the benchmark owner is the party being asked to *rely* on the
certificate, so it should hold a veto over the policy that produces it — which JAX leaves were
allow-listed, which lines were declared private. Approving means *"I accept this as sufficient
evidence."*

We approve one at a time to show the gate holding.

In [ ]:
model_owner.sync()
benchmark_owner.sync()

print(f"  Model owner     sees status={model_owner.jobs['restrict_engine_review'].status}")
print(f"  Benchmark owner sees status={benchmark_owner.jobs['restrict_engine_review'].status}")

In [ ]:
# Inspect the submitted job before approving
model_owner.jobs["restrict_engine_review"]

In [ ]:
# Model owner approves first — the job must still NOT be runnable.
model_owner.approve_job(model_owner.jobs["restrict_engine_review"])
enclave.sync()
status_after_one = enclave.jobs["restrict_engine_review"].status
print(f"  Model owner approved     → enclave status: {status_after_one}")
assert status_after_one == "pending", f"expected 'pending', got '{status_after_one}'"
print("    ...still gated on the benchmark owner")

# Benchmark owner approves → both votes are in.
benchmark_owner.approve_job(benchmark_owner.jobs["restrict_engine_review"])
enclave.sync()
status_after_both = enclave.jobs["restrict_engine_review"].status
print(f"  Benchmark owner approved → enclave status: {status_after_both}")
assert status_after_both == "approved", f"expected 'approved', got '{status_after_both}'"
print("    Both approvals received — restrict job is APPROVED")

## Step 8 — Enclave runs `syft-restrict` on the private engine

In [ ]:
%%time
enclave.run_jobs()

restrict_job = enclave.jobs["restrict_engine_review"]
print(f"  Enclave job status: {restrict_job.status}")
assert restrict_job.status == "done", (
    f"restrict job did not complete: {restrict_job.status}\n"
    f"check restrict_job.stderr for details"
)
print(f"  Output files: {[p.name for p in restrict_job.output_paths]}")

In [ ]:
# What syft-restrict reported, from inside the enclave
print(restrict_job.stdout)

## Step 9 — Share the verdict with the benchmark owner

Because the benchmark owner is a key in the `datasets` dict (with an empty list),
`distribute_results()` delivers the certificate and obfuscated engine to it.

In [ ]:
enclave.distribute_results()
benchmark_owner.sync()

bo_outputs = benchmark_owner.jobs["restrict_engine_review"].output_paths
print(f"  Benchmark owner received : {[p.name for p in bo_outputs]}")
assert len(bo_outputs) > 0, "benchmark owner did not receive the restrict outputs"
print()
print("  Certificate + obfuscated engine delivered to the benchmark owner")
print("  (it never gained access to gemma3_model itself — only to the verdict about it)")

## Step 10 — Benchmark owner inspects the evidence

In [ ]:
benchmark_owner.sync()
bo_restrict_job = benchmark_owner.jobs["restrict_engine_review"]
outputs = {p.name: p for p in bo_restrict_job.output_paths}

certificate = json.loads(outputs["gemma_inference.certificate.json"].read_text())

print("  CERTIFICATE (what the enclave attests)")
print("  " + "─" * 68)
for k, v in certificate.items():
    print(f"    {k:18}: {v}")

report = json.loads(outputs["restrict_report.json"].read_text())
print()
print(f"  Verdict    : {'PASSED' if report['ok'] else 'FAILED'}")
print(f"  Violations : {len(report['violations'])}")

In [ ]:
# The obfuscated artifact the benchmark owner is allowed to read.
obf = outputs["gemma_inference.obfuscated.py"].read_text()
lines = obf.splitlines()

# The model configs — obfuscated: every layer count, dim and RoPE base blanked to ■.
_cfg = next(i for i, l in enumerate(lines) if l.startswith("░v0 = {"))
print("── the model configs (obfuscate) " + "─" * 42)
print("\n".join(lines[_cfg:_cfg + 12]))
print("    ...")

# A Flax module — signature obfuscated, body hidden.
_cls = next(i for i, l in enumerate(lines) if l.startswith("class ░Cls1"))
_end = next(i for i, l in enumerate(lines[_cls:], _cls) if "obfuscate-end" in l)
print()
print("── a module: skeleton visible, body hidden " + "─" * 32)
print("\n".join(lines[_cls:_end]))

In [ ]:
# ...while the public region is copied through byte-for-byte — the data owners read this in full.
_pub = next(i for i, l in enumerate(lines) if l.startswith("def _get("))
print("── the public wrappers (verbatim) " + "─" * 41)
print("\n".join(lines[_pub:_pub + 14]))

### What the benchmark owner just learned — and what it didn't

**Learned** (the public region, byte-for-byte):
- prompts reach the model through a tokenizer and generation loop it can read in full
- every value leaving the private region passes back through that same public code
- the wrappers the private region depends on — `_get`, `shape_of`, `append_to` — are right there
- from the **certificate**: the source hash, the policy id, and that all 81 calls resolved to the
  23 allow-listed JAX/Flax leaves. No file IO, no network, no `eval`/`exec`/`getattr`, no `import`,
  no decorators, no f-strings in the private region.

**Not learned** (the private region):
- `░v0 = {"■": dict(░v1=■, ...)}` — the configs: layer counts, embedding dims, head counts,
  sliding-window sizes, RoPE bases
- `class ░Cls2(nn.Module)` + `■■■■■■■■` — the class skeleton is visible, every body is not

Note the division of labour between the two tiers. `obfuscate` leaves the skeleton legible;
`hide` blanks the bodies — and the bodies are where the architecture actually lives. Had the
bodies merely been obfuscated, `jnp.mean(jnp.square(x)) * jax.lax.rsqrt(...)` would still read as
RMSNorm to anyone who knows the field. The assurance therefore comes from the **certificate and the
policy the benchmark owner approved**, not from reading the math — which is the honest arrangement:
the benchmark owner vetted the *rules*, the enclave vouches that the code obeys them.

### What this certificate does *not* prove

1. **The output is itself a leak channel** (README §5.1). `syft-restrict` proves the engine only does
   math; it does **not** prove the *tokens it emits* carry no private bits. A malicious model could
   encode prompts into its completions. That needs an orthogonal control — output schema review, DP
   noise, rate limiting — and no AST checker can close it.

2. **The trusted libraries are trusted.** The policy constrains the *caller's* code; a malicious
   `jax`/`flax`/`orbax` build defeats it entirely (README §5.4). Attest the library versions.

3. **Timing and cache side channels** are outside the model entirely (README §5.2).

---
# Part 2 — The inference job

The review is done. The benchmark owner now takes the researcher's part: it browses the model owner's
mock, writes the job, and submits it.

## Step 11 — Benchmark owner browses the mock datasets

In [ ]:
benchmark_owner.sync()
benchmark_owner.datasets.get_all()


In [ ]:
print(benchmark_owner.datasets[0].mock_files[0].read_text())


## Step 12 — Benchmark owner submits the inference job

Identical to `1. enclave_gemma_inmem.ipynb` — it loads the model owner's private engine by name and
calls `gemma.generate(...)`. The engine it loads is the one the enclave just certified.

In [ ]:
# mock_jax=True emits placeholder completions instead of loading Gemma / running JAX.
# Set to False to run the real inference engine. Only JOB_CODE changes.
mock_jax = True

if mock_jax:
    JOB_CODE = f'''
import csv
import json
import os

import syft_client as sc

print("mock_jax=True -- emitting placeholder completions (no model load, no JAX)")

# Load Benchmark owner's private benchmark (AILuminate CSV) and iterate over its rows
prompt_path = sc.resolve_dataset_file_path(
    "safety_prompts", owner_email="{benchmark_owner.email}"
)
with open(prompt_path, newline="") as f:
    prompt_rows = list(csv.DictReader(f))
print(f"Loaded {{len(prompt_rows)}} evaluation prompts")

results = []
for i, row in enumerate(prompt_rows):
    prompt = row["prompt_text"]
    prompt_id = row["prompt_uid"]
    print(f"  [{{i+1}}/{{len(prompt_rows)}}] {{prompt_id}}: {{prompt[:50]}}...")
    results.append({{
        "prompt_id": prompt_id,
        "prompt": prompt,
        "completion": "[mock completion]",
        "ttft": 0.0,
        "decode_tps": 0.0,
    }})

os.makedirs("outputs", exist_ok=True)
with open("outputs/safety_eval_results.json", "w") as f:
    json.dump({{
        "model": "mock",
        "total_prompts": len(results),
        "results": results,
    }}, f, indent=2)

print(f"\\nInference complete (mock). {{len(results)}} prompts evaluated.")
'''
else:
    JOB_CODE = f'''
import csv
import json
import os

import syft_client as sc

# Resolve Model owner's private model dataset directory
model_files = sc.resolve_dataset_files_path(
    "gemma3_model", owner_email="model_owner@openmined.org"
)
weights_dir = str(model_files[0].parent)

# Import the inference module from the model owner's private dataset
gemma = sc.load_dataset_code(
    "gemma3_model.gemma_inference", owner_email="model_owner@openmined.org"
)

# Load model, tokenizer, and params in one call
print(f"Loading Gemma 3 {MODEL_SIZE.upper()}-IT from {{weights_dir}}...")
model, tokenizer, params = gemma.setup_model("{MODEL_SIZE}", weights_dir)
print("Model loaded successfully")

# Load Benchmark owner's private benchmark (AILuminate CSV) and iterate over its rows
prompt_path = sc.resolve_dataset_file_path(
    "safety_prompts", owner_email="benchmark_owner@openmined.org"
)
with open(prompt_path, newline="") as f:
    prompt_rows = list(csv.DictReader(f))
print(f"Loaded {{len(prompt_rows)}} evaluation prompts")

# Run inference on each prompt
results = []
for i, row in enumerate(prompt_rows):
    prompt = row["prompt_text"]
    prompt_id = row["prompt_uid"]
    print(f"  [{{i+1}}/{{len(prompt_rows)}}] {{prompt_id}}: {{prompt[:50]}}...")
    completion, stats = gemma.generate(
        model, params, tokenizer, prompt,
        max_new_tokens=100, temperature=0.8, top_k=40,
    )
    results.append({{
        "prompt_id": prompt_id,
        "prompt": prompt,
        "completion": completion,
        "ttft": stats["ttft"],
        "decode_tps": stats["decode_tps"],
    }})

# Write outputs
os.makedirs("outputs", exist_ok=True)
with open("outputs/safety_eval_results.json", "w") as f:
    json.dump({{
        "model": "{CKPT_SUBDIR}",
        "total_prompts": len(results),
        "results": results,
    }}, f, indent=2)

print(f"\\nInference complete. {{len(results)}} prompts evaluated.")
'''


GEMMA_DEPS = ["jax[cpu]", "flax", "orbax-checkpoint", "sentencepiece"]

code_path = create_code_file(JOB_CODE)

benchmark_owner.submit_python_job(
    enclave.email,
    code_path,
    "safety_eval_job",
    datasets={
        model_owner.email: ["gemma3_model"],
        benchmark_owner.email: ["safety_prompts"],
    },
    share_results_with_do=False,
    dependencies=GEMMA_DEPS,
)

print(f"  Job 'safety_eval_job' submitted to enclave by {benchmark_owner.email}")
print(f"  Dependencies: {GEMMA_DEPS}")

## Step 13 — Enclave distributes; both owners approve

The benchmark owner approves a job it submitted itself — the gate is there to protect the *model
owner's* assets too, so both votes are still required.

In [ ]:
%%time
enclave.sync()
enclave.receive_jobs()
print(f"  Enclave job status : {enclave.jobs['safety_eval_job'].status}")

model_owner.sync()
benchmark_owner.sync()

model_owner.approve_job(model_owner.jobs["safety_eval_job"])
print("  Model owner approved")

benchmark_owner.approve_job(benchmark_owner.jobs["safety_eval_job"])
print("  Benchmark owner approved")

enclave.sync()
eval_status = enclave.jobs["safety_eval_job"].status
print(f"  Enclave job status: {eval_status}")
assert eval_status == "approved"
print("  Both approvals received — job is APPROVED")

## Step 14 — Enclave executes the inference job

The enclave runs Gemma 3 against the model owner's private weights and the benchmark owner's private
prompts. This is the slow one: it builds a venv with the full JAX stack, loads the checkpoint, and
decodes every prompt.

In [ ]:
%%time
enclave.run_jobs()

eval_job = enclave.jobs["safety_eval_job"]
print(f"  Enclave job status: {eval_job.status}")
assert eval_job.status == "done", f"job failed: {eval_job.status}"
print("  Job completed successfully")

enclave.distribute_results()
print("  Results distributed")

## Step 15 — Benchmark owner retrieves and inspects results

In [ ]:
%%time
benchmark_owner.sync()

eval_job = benchmark_owner.jobs["safety_eval_job"]
print(f"  Benchmark owner job status : {eval_job.status}")
assert eval_job.status == "done"
assert len(eval_job.output_paths) > 0

with open(eval_job.output_paths[0]) as f:
    result = json.load(f)

print(f"\n  Model: {result['model']}")
print(f"  Total prompts evaluated: {result['total_prompts']}")
print()

for r in result["results"]:
    print(f"  prompt_id  : {r['prompt_id']}")
    print(f"  prompt     : {r['prompt']}")
    completion = r["completion"]
    print(f"  completion : {completion[:120]}..." if len(completion) > 120 else f"  completion : {completion}")
    print(f"  TTFT={r['ttft']:.2f}s  decode={r['decode_tps']:.1f} tok/s")
    print()

## Step 16 — Model owner is denied the results

The inference job was submitted with `share_results_with_do=False`. The model owner contributed the
private weights and engine, but the output is shared only with the benchmark owner who submitted the
job — we assert the model owner sees **no** output files.

In [ ]:
%%time
model_owner.sync()

mo_job = model_owner.jobs["safety_eval_job"]
mo_outputs = [p.name for p in mo_job.output_paths]
print(f"  Model owner — output files : {mo_outputs}")

# share_results_with_do=False — the model owner contributed the weights/engine but must NOT
# see the job output; only the benchmark owner (the submitter) receives the results.
assert len(mo_outputs) == 0, f"model owner must not receive job output, got {mo_outputs}"
print()
print("  share_results_with_do=False — the model owner cannot see the job output")

---
## Summary

| Step | Actor | Action | Outcome |
|------|-------|--------|---------|
| 1 | Model owner | Upload Gemma 3 (model card mock + private weights/**engine**) | Architecture never leaves the owner |
| 2 | Benchmark owner | Upload AI safety prompts | Prompts available |
| 3 | Both | Share private data with enclave | Enclave can access both assets |
| **4** | **Model owner** | **Submit `syft-restrict` job** (no jax needed) | **Enclave asked to vouch for an unreadable engine** |
| **5** | **Both owners** | **`approve_job()`** | **Gate held: one approval → still `pending`** |
| **6** | **Enclave** | **`run_jobs()` → `restrict.run(...)`** | **78 calls checked, PASSED** |
| **7** | **Enclave** | **`distribute_results()`** | **Certificate + obfuscated engine → benchmark owner** |
| 8 | **Benchmark owner** | Submit inference job (full jax stack) | Job sent to enclave |
| 9 | Both owners | `approve_job()` | Status → **approved** |
| 10 | Enclave | `run_jobs()` | Gemma 3 runs on private weights + prompts |
| 11 | Benchmark owner | `sync()` + read output | Only the submitter receives results (`share_results_with_do=False`) |

The Benchmark Owner wears both hats — data owner and data scientist — because `login_do` grants the DO
and DS roles together. Nothing else in the flow changes: the enclave still gates every job on **both**
data owners.

### The trust argument, end to end

| Asset | Who sees it | Why the others are OK with that |
|-------|-------------|--------------------------------|
| Gemma 3 weights | enclave | never leave the enclave |
| Safety prompts | enclave | never leave the enclave |
| **Architecture** | **enclave** | **`syft-restrict` certificate proves the engine only does allow-listed math** |
| Outputs | benchmark owner only | `share_results_with_do=False` — the model owner approved the job but is never shown its output |


---